# Churn Prediction Inference - Batch or serverless real-time


With AutoML, our best model was automatically saved in our MLFlow registry.

All we need to do now is use this model to run Inferences. A simple solution is to share the model name to our Data Engineering team and they'll be able to call this model within the pipeline they maintained. That's what we did in our Spark Declarative Pipelines pipeline!

Alternatively, this can be schedule in a separate job. Here is an example to show you how MLFlow can be directly used to retriver the model and run inferences.

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F04-Data-Science-ML%2F04.3-running-inference&demo_name=lakehouse-retail-c360&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-retail-c360%2F04-Data-Science-ML%2F04.3-running-inference&version=1">

In [0]:
%pip install mlflow==3.1.0
dbutils.library.restartPython()

  Using cached mlflow-3.1.0-py3-none-any.whl.metadata (29 kB)
  Using cached mlflow_skinny-3.1.0-py3-none-any.whl.metadata (30 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached graphql_relay-3.2.0-py3-none-any.whl.metadata (12 kB)
Using cached mlflow-3.1.0-py3-none-any.whl (24.7 MB)
Using cached mlflow_skinny-3.1.0-py3-none-any.whl (1.9 MB)
Using cached docker-7.1.0-py3-none-any.whl (147 kB)
Using cached graphene-3.4.3-py2.py3-none-any.whl (114 kB)
Using cached graphql_relay-3.2.0-py3-none-any.whl (16 kB)
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 2.21.3
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-4311a605-a91a-4cd6-a004-b257abf9fe9d
    Can't uninstall 'mlflow-skinny'. No files were found to uninstall.
ERROR: pip's dependency resolver does

In [0]:
%run ../_resources/00-setup $reset_all_data=false

USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_retail_c360`


data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.


##Deploying the model for batch inferences

Now that our model is available in the Registry, we can load it to compute our inferences and save them in a table to start building dashboards.

We will use MLFlow function to load a pyspark UDF and distribute our inference in the entire cluster. If the data is small, we can also load the model with plain python and use a pandas Dataframe.

### Scaling inferences using Spark 
We'll first see how it can be loaded as a spark UDF and called directly in a SQL function:

In [0]:
import mlflow
model_name = "dbdemos_customer_churn"
mlflow.set_registry_uri("databricks-uc")
#                                                                                                Alias
#                                                                                  Model name       |
#                                                                                        |          |
predict_churn_udf = mlflow.pyfunc.spark_udf(spark, model_uri=f"models:/{catalog}.{db}.{model_name}@prod", env_manager='virtualenv', result_type='long')
# Note: virtualenv will recreate an env from scratch which can take some time, but prevent any version issue. If you're using the same compute as for training, you can remove it to use the local env instead (just install the lib from the requirements.txt file as below)
#We can use the function in SQL
spark.udf.register("predict_churn", predict_churn_udf)

2025/12/09 16:56:34 INFO mlflow.pyfunc: This UDF will use virtualenv to recreate the model's software environment for inference. This may take extra time during execution.


2025/12/09 16:56:34 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2025/12/09 16:56:34 INFO mlflow.utils.virtualenv: Installing python 3.12.3 if it does not exist
2025/12/09 16:57:49 INFO mlflow.utils.virtualenv: Creating a new environment in /local_disk0/.ephemeral_nfs/repl_tmp_data/ReplId-19b03-f841a-1/mlflow/envs/virtualenv_envs/mlflow-2e7e6045218a336288d5eded6dc6233a3bccf3c4 with /local_disk0/.ephemeral_nfs/repl_tmp_data/ReplId-19b03-f841a-1/mlflow/envs/pyenv_root/versions/3.12.3/bin/python
2025/12/09 16:57:49 INFO mlflow.utils.virtualenv: Installing dependencies
2025/12/09 16:58:41 INFO mlflow.utils.environment: === Running command '['bash', '-c', 'source /local_disk0/.ephemeral_nfs/repl_tmp_data/ReplId-19b03-f841a-1/mlflow/envs/virtualenv_envs/mlflow-2e7e6045218a336288d5eded6dc6233a3bccf3c4/bin/activate && python -c ""']'


In [0]:
columns = predict_churn_udf.metadata.get_input_schema().input_names()
spark.table('churn_features').withColumn("churn_prediction", predict_churn_udf(*columns)).display()

user_id,email,creation_date,last_activity_date,firstname,lastname,address,canal,country,gender,age_group,churn,order_count,total_amount,total_item,last_transaction,platform,event_count,session_count,last_event,days_since_creation,days_since_last_activity,days_last_event,churn_prediction
ccbd668a-50b1-485c-bb82-7b994fed0fd7,b69acaa2f1f2a41fec422827d6604e4b9c8d6e5c,2021-10-10T00:00:00Z,2023-06-08T14:21:29Z,Russell,Jordan,"283 John Creek Suite 938 Amberchester, LA 59056",WEBAPP,FR,1,6,0,3,161,5,2023-06-07T20:26:46Z,ios,3,3,2023-06-03T08:50:53Z,1521,915,920,0
509b3da6-c25a-4b9a-b375-5ade442b95c2,8d7dc00756b73ec682b43973616335167ea3d23b,2022-03-13T00:00:00Z,2023-06-04T23:26:33Z,Jeffrey,Walker,"48351 Colleen Drives Suite 373 Danielborough, LA 16110",WEBAPP,SPAIN,1,5,0,2,103,5,2023-06-08T21:50:54Z,ios,2,1,2023-06-07T17:39:33Z,1367,919,916,0
a46211c0-d5d5-4067-9993-f23555724281,88ee42714f43e273f0deb5c29bd987e4a80f452c,2022-01-02T00:00:00Z,2023-06-08T19:56:16Z,William,Romero,"3501 Wright Ridge Suite 542 Johnsonside, MP 03305",WEBAPP,USA,1,1,1,2,63,3,2023-06-04T10:09:18Z,android,2,2,2023-06-08T07:17:45Z,1437,915,915,1
dddede45-aa29-4a2a-abb0-e6b5259a72b2,1979d5edc327d774e692a66d3beb6cd5221dcfc5,2022-04-07T00:00:00Z,2023-06-03T17:02:10Z,Daniel,Garcia,"79150 Bush Plaza Apt. 420 Jeffreychester, NY 79219",WEBAPP,USA,0,2,1,3,202,8,2023-06-05T04:53:53Z,ios,3,3,2023-06-01T00:35:00Z,1342,920,922,1
9691861e-23fd-4ce4-b56a-e4ed0070c483,b858e3e25de747418e7477c102ff1c4963aa3608,2021-11-09T00:00:00Z,2023-06-06T05:40:51Z,Ashley,Armstrong,"32279 Roberta Vista Suite 794 South Brittany, MS 77487",MOBILE,USA,1,6,1,4,146,7,2023-06-09T13:48:07Z,ios,4,4,2023-06-06T19:36:01Z,1491,917,917,1
868381d4-be16-4bdb-a5d4-3fc5954933ae,3672bdef0aaab8963959917d3011b26c1e7684b0,2021-09-02T00:00:00Z,2023-06-09T08:00:29Z,Keith,Roberts,"8171 Veronica Roads Mileshaven, VA 93489",WEBAPP,FR,0,1,1,3,125,5,2023-06-05T17:52:29Z,other,3,3,2023-06-08T10:45:26Z,1559,914,915,1
f3887f37-b523-400a-9bf7-584d39eee42a,80d793b729fc8f2b95af6523b6e0475609ac7a6d,2022-04-22T00:00:00Z,2023-06-03T02:47:44Z,James,Gonzalez,"272 Adam Rapids Suite 673 Port Jacob, NH 01431",WEBAPP,USA,1,6,0,2,87,5,2023-06-09T15:32:55Z,ios,2,2,2023-06-08T02:04:30Z,1327,920,915,1
2f722ef9-a136-4300-9c53-5014897b93fb,cdcd7cd6eb2e9cdd2a6df9e5e5f51a149c10ce27,2022-03-24T00:00:00Z,2023-06-08T19:19:42Z,Andrea,Arnold,"0568 Karen Fields Codyfurt, FL 58814",MOBILE,SPAIN,1,1,1,1,56,2,2023-06-09T19:13:29Z,other,1,1,2023-06-08T21:47:53Z,1356,915,915,1
c2d16387-be68-4608-acc5-e6c20f291a2d,532adff3753657858811658049ae788ae7ae2f08,2022-06-30T00:00:00Z,2023-06-09T14:25:22Z,Jeffrey,Walker,"5173 Vance Ports Apt. 505 Lukemouth, IA 75436",WEBAPP,USA,1,1,1,5,249,9,2023-06-08T19:44:38Z,ios,5,5,2023-06-01T05:17:42Z,1258,914,922,1
27cd9d49-c013-4ab4-8b23-aefe7e8806ce,17226fdf64d31647b70cd6d4067b436369d5e5a9,2022-09-05T00:00:00Z,2023-06-06T16:19:32Z,Sarah,Hebert,"PSC 8021, Box 5675 APO AP 12067",PHONE,USA,1,7,0,7,408,13,2023-06-06T10:30:11Z,ios,7,7,2023-06-07T07:57:39Z,1191,917,916,1


### Pure pandas inference
If we have a small dataset, we can also compute our segment using a single node and pandas API:

In [0]:
from mlflow.store.artifact.models_artifact_repo import ModelsArtifactRepository
import mlflow
# Use the Unity Catalog model registry
mlflow.set_registry_uri("databricks-uc")
# download model requirement from remote registry
requirements_path = ModelsArtifactRepository(f"models:/{catalog}.{db}.dbdemos_customer_churn@prod").download_artifacts(artifact_path="requirements.txt") 

In [0]:
%pip install -r $requirements_path
dbutils.library.restartPython()

  Using cached pandas-2.2.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.2.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.7 MB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
  Attempting uninstall: pandas
    Found existing installation: pandas 1.5.3
    Not uninstalling pandas at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-4311a605-a91a-4cd6-a004-b257abf9fe9d
    Can't uninstall 'pandas'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ../_resources/00-setup $reset_all_data=false

USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_retail_c360`


data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.


In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")
model_name = "dbdemos_customer_churn"
model = mlflow.pyfunc.load_model(f"models:/{catalog}.{db}.{model_name}@prod")
columns = model.metadata.get_input_schema().input_names()
df = spark.table('churn_features').select(*columns).limit(10).toPandas()
df['churn_prediction'] = model.predict(df)
df.head(3)

[LightGBM] [Warning] lambda_l2 is set=117.97769127793491, reg_lambda=0.0 will be ignored. Current value: lambda_l2=117.97769127793491
[LightGBM] [Warning] lambda_l1 is set=0.24114681074974328, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.24114681074974328


,user_id,canal,country,gender,age_group,order_count,total_amount,total_item,last_transaction,platform,event_count,session_count,days_since_creation,days_since_last_activity,days_last_event,churn_prediction
0,ccbd668a-50b1-485c-bb82-7b994fed0fd7,WEBAPP,FR,1,6,3,161,5,2023-06-07 20:26:46,ios,3,3,1521,915,920,0
1,509b3da6-c25a-4b9a-b375-5ade442b95c2,WEBAPP,SPAIN,1,5,2,103,5,2023-06-08 21:50:54,ios,2,1,1367,919,916,0
2,a46211c0-d5d5-4067-9993-f23555724281,WEBAPP,USA,1,1,2,63,3,2023-06-04 10:09:18,android,2,2,1437,915,915,1



## Realtime model serving with Databricks serverless serving

<img style="float: right; margin-left: 20px" width="700" src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/retail/lakehouse-churn/lakehouse-c360-model-serving.png?raw=true" />

Databricks also provides serverless serving.

Click on model Serving, enable realtime serverless and your endpoint will be created, providing serving over REST api within a Click.

Databricks Serverless offer autoscaling, including downscaling to zero when you don't have any traffic to offer best-in-class TCO while keeping low-latencies model serving.

To deploy your serverless model, open the [Model Serving menu](https://xxxx.cloud.databricks.com/?o=1660015457675682#mlflow/endpoints), and select the model you registered within Unity Catalog.

In [0]:
from mlflow.deployments import get_deploy_client
model_endpoint_name = "dbdemos_customer_churn_endpoint"
last_version = get_last_model_version(f"{catalog}.{db}.{model_name}")
client = get_deploy_client("databricks")
try:
    endpoint = client.create_endpoint(
        name=model_endpoint_name,
        config={
            "served_entities": [
                {
                    "name": f"dbdemos_customer_churn_endpoint_{last_version}",
                    "entity_name": f"{catalog}.{db}.{model_name}",
                    "entity_version": last_version,
                    "workload_size": "Small",
                    "scale_to_zero_enabled": True
                }
            ]
        }
    )
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint {catalog}.{db}.{model_endpoint_name} already exists. Skipping creation.")
    else:
        raise e

while client.get_endpoint(model_endpoint_name)['state']['config_update'] == 'IN_PROGRESS':
    time.sleep(10)

Endpoint main.dbdemos_retail_c360.dbdemos_customer_churn_endpoint already exists. Skipping creation.


In [0]:
dataset = spark.table('churn_features').select(*columns).limit(3).toPandas()
#Make it a string to send to the inference endpoint
dataset['last_transaction'] = dataset['last_transaction'].astype(str)
dataset

,user_id,canal,country,gender,age_group,order_count,total_amount,total_item,last_transaction,platform,event_count,session_count,days_since_creation,days_since_last_activity,days_last_event
0,ccbd668a-50b1-485c-bb82-7b994fed0fd7,WEBAPP,FR,1,6,3,161,5,2023-06-07 20:26:46,ios,3,3,1521,915,920
1,509b3da6-c25a-4b9a-b375-5ade442b95c2,WEBAPP,SPAIN,1,5,2,103,5,2023-06-08 21:50:54,ios,2,1,1367,919,916
2,a46211c0-d5d5-4067-9993-f23555724281,WEBAPP,USA,1,1,2,63,3,2023-06-04 10:09:18,android,2,2,1437,915,915


In [0]:
from mlflow import deployments

def score_model(dataset):
  client = mlflow.deployments.get_deploy_client("databricks")
  payload = {"dataframe_split": dataset.to_dict(orient='split')}
  predictions = client.predict(endpoint=model_endpoint_name, inputs=payload)
  print(predictions)

#Deploy your model and uncomment to run your inferences live!
score_model(dataset)

{'predictions': [0, 0, 1]}



# Next step: Leverage inferences and automate actions to increase revenue

## Automate action to reduce churn based on predictions

We now have an end 2 end data pipeline analizing and predicting churn. We can now easily trigger actions to reduce the churn based on our business:

- Send targeting email campaign to the customer the most likely to churn
- Phone campaign to discuss with our customers and understand what's going
- Understand what's wrong with our line of product and fixing it

These actions are out of the scope of this demo and simply leverage the Churn prediction field from our ML model.

## Track churn impact over the next month and campaign impact

Of course, this churn prediction can be re-used in our dashboard to analyse future churn and measure churn reduction. 

The pipeline created with the Lakehouse will offer a strong ROI: it took us a few hours to setup this pipeline end 2 end and we have potential gain for $129,914 / month!

<img width="800px" src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/lakehouse-retail-churn-dbsql-prediction-dashboard.png">

<a dbdemos-dashboard-id="churn-prediction" href='/sql/dashboardsv3/01f1b2d441601962affad7fdce91ca8e'>Open the Churn prediction DBSQL dashboard</a>





## Reducing churn leveraging Databricks GenAI and LLMs capabilities 

GenAI provides unique capabilities to improve your customer relationship, providing better services but also better analyzing your churn risk.

Databricks provides built-in GenAI capabilities for you to accelerate such GenAI apps deployment. 

Discover how with the [Agent Tools]($../05-Generative-AI/05.1-Agent-Functions-Creation) Notebook in the new Generative AI section of this demo!

[Go back to the introduction]($../00-churn-introduction-lakehouse)